# Tutorial 07 — Тензор роста и мультипликативное разложение

Notebook вычисляет все результаты через локальный пакет и не загружает сохранённые рисунки.

## 1. Среда и импорт

In [ ]:
LANGUAGE = "ru"

import importlib.util
import os
import sys
from pathlib import Path


def _is_repository_root(path: Path) -> bool:
    return (path / "pyproject.toml").is_file() and (
        path / "src" / "biomechanics_tutorials"
    ).is_dir()


def _find_repository_root() -> Path:
    candidates = []
    installed_spec = importlib.util.find_spec("biomechanics_tutorials")
    if installed_spec is not None and installed_spec.origin:
        package_file = Path(installed_spec.origin).resolve()
        if len(package_file.parents) >= 3:
            candidates.append(package_file.parents[2])
    environment_root = os.environ.get("BMRT_ROOT")
    if environment_root:
        candidates.append(Path(environment_root).expanduser())
    current = Path.cwd().resolve()
    candidates.extend([current, *current.parents])
    home = Path.home()
    for directory in [home, home / "Desktop", home / "Documents", home / "Downloads"]:
        candidates.append(directory / "Biomechanics-Research-Tutorials")
        if directory.is_dir():
            candidates.extend(directory.glob("Biomechanics-Research-Tutorials*"))
    for candidate in candidates:
        candidate = candidate.resolve()
        if _is_repository_root(candidate):
            return candidate
    raise RuntimeError(
        "Repository root was not found. Extract the complete repository or install it with python -m pip install -e .[dev]"
    )


REPOSITORY_ROOT = _find_repository_root()
SOURCE_DIRECTORY = REPOSITORY_ROOT / "src"
if str(SOURCE_DIRECTORY) not in sys.path:
    sys.path.insert(0, str(SOURCE_DIRECTORY))

import matplotlib.pyplot as plt
import numpy as np
import biomechanics_tutorials

from biomechanics_tutorials.growth_kinematics import (
    GrowthLawParameters,
    GrowthMaterialParameters,
    commutator_norm,
    compose_growth_increments,
    finite_difference_first_piola,
    growth_tensor_isotropic,
    growth_tensor_orthotropic,
    growth_tensor_transversely_isotropic,
    incompatibility_norm_2d,
    jacobian_bookkeeping,
    material_basis,
    multiplicative_decomposition,
    reduced_strip_equilibrium,
    rotation_matrix,
    simulate_stress_driven_growth,
    total_first_piola,
    total_reference_energy,
)
from biomechanics_tutorials.plotting import apply_tutorial_style

apply_tutorial_style()
print("repository:", REPOSITORY_ROOT)
print("python:", sys.executable)
print("local package:", Path(biomechanics_tutorials.__file__).resolve())


## 2. Мультипликативное разложение и баланс определителей

In [ ]:
F = np.diag([1.20, 0.95, 0.90])
Fg = np.diag([1.08, 1.02, 0.97])
Fe, _ = multiplicative_decomposition(F, Fg)
values = jacobian_bookkeeping(F, Fg)
print("Fe =\n", Fe)
print(values)
print("reconstruction error:", np.linalg.norm(F - Fe @ Fg))

## 3. Семейства тензоров роста

In [ ]:
basis = material_basis([1.0, 1.0, 0.0], [0.0, 0.0, 1.0])
tensors = {
    "isotropic": growth_tensor_isotropic(linear_growth=1.15),
    "fiber": growth_tensor_transversely_isotropic(1.30, 1.0),
    "transverse": growth_tensor_transversely_isotropic(1.0, 1.18),
    "orthotropic": growth_tensor_orthotropic([1.30, 1.10, 0.92], basis),
}
for name, tensor in tensors.items():
    print(name, "Jg=", np.linalg.det(tensor), "principal=", np.linalg.eigvalsh(tensor))

## 4. Свободный и стеснённый рост

In [ ]:
growth_values = np.linspace(0.85, 1.25, 120)
free_energy = []
constrained_energy = []
for value in growth_values:
    growth = growth_tensor_isotropic(linear_growth=value)
    free_energy.append(total_reference_energy(growth, growth))
    constrained_energy.append(total_reference_energy(np.eye(3), growth))
plt.plot(growth_values, free_energy, label="free")
plt.plot(growth_values, constrained_energy, label="constrained")
plt.xlabel("growth stretch")
plt.ylabel("energy")
plt.legend();

## 5. Верификация определяющего соотношения

In [ ]:
material = GrowthMaterialParameters(shear_modulus=1.4, bulk_modulus=20.0)
F = np.array([[1.16, 0.10, 0.0], [0.01, 0.97, 0.03], [0.0, 0.0, 0.93]])
Fg = np.diag([1.08, 0.98, 1.02])
P = total_first_piola(F, Fg, material)
P_fd = finite_difference_first_piola(F, Fg, material)
print("maximum stress derivative error:", np.max(np.abs(P - P_fd)))
angles = np.linspace(0.0, 2.0 * np.pi, 100)
energies = [total_reference_energy(rotation_matrix([0.2, 0.7, 0.3], angle) @ F, Fg, material) for angle in angles]
print("frame-indifference range:", np.ptp(energies))

## 6. Однородный рост, управляемый напряжением

In [ ]:
time = np.linspace(0.0, 35.0, 1401)
stretch = 1.35
F = np.diag([stretch, 1.0 / np.sqrt(stretch), 1.0 / np.sqrt(stretch)])
result = simulate_stress_driven_growth(
    time,
    F,
    law=GrowthLawParameters(rate=0.10, mode="diagonal", response_limit=0.16),
)
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
axes[0].plot(time, result["Fg"][:, 0, 0], label="Fg11")
axes[0].plot(time, result["Fg"][:, 1, 1], label="Fg22")
axes[0].legend()
axes[1].plot(time, result["mandel"][:, 0, 0], label="M11")
axes[1].plot(time, result["mandel"][:, 1, 1], label="M22")
axes[1].legend()
axes[2].semilogy(time, np.maximum(result["energy"], 1e-12))
axes[2].set_ylabel("energy")
for ax in axes:
    ax.set_xlabel("time")
plt.tight_layout();

## 7. Чувствительность к скорости

In [ ]:
rates = [0.03, 0.07, 0.14]
for rate in rates:
    trajectory = simulate_stress_driven_growth(
        time, F,
        law=GrowthLawParameters(rate=rate, mode="diagonal", response_limit=0.16),
    )
    plt.plot(time, trajectory["mandel"][:, 0, 0], label=f"k={rate}")
plt.xlabel("time")
plt.ylabel("M11")
plt.legend();

## 8. Некоммутирующие анизотропные пути роста

In [ ]:
first = growth_tensor_orthotropic([1.22, 0.94, 1.0])
angles = np.linspace(0.0, 90.0, 91)
differences = []
for angle in angles:
    q = rotation_matrix([0.0, 0.0, 1.0], np.deg2rad(angle))
    second = growth_tensor_orthotropic([1.16, 0.91, 1.0], q)
    path_ab = compose_growth_increments([first, second])
    path_ba = compose_growth_increments([second, first])
    differences.append(np.linalg.norm(path_ab - path_ba))
plt.plot(angles, differences)
plt.xlabel("relative angle, degrees")
plt.ylabel("path difference")
print("commutator at 45 degrees:", commutator_norm(first, growth_tensor_orthotropic([1.16, 0.91, 1.0], rotation_matrix([0,0,1], np.pi/4))))

## 9. Диагностика пространственной несовместности

In [ ]:
y, x = np.mgrid[-1.0:1.0:101j, -1.0:1.0:101j]
theta = 1.0 + 0.22 * np.exp(-3.5 * (x**2 + y**2)) + 0.05 * x
field = np.zeros(theta.shape + (2, 2))
field[..., 0, 0] = theta
field[..., 1, 1] = theta
curl_norm = incompatibility_norm_2d(field, x[0, 1]-x[0, 0], y[1, 0]-y[0, 0])
fig, axes = plt.subplots(1, 2, figsize=(9, 3.8))
axes[0].imshow(theta, origin="lower", cmap="viridis")
axes[0].set_title("growth stretch")
axes[1].imshow(curl_norm, origin="lower", cmap="magma")
axes[1].set_title("incompatibility diagnostic")
for ax in axes:
    ax.set_xticks([])
    ax.set_yticks([])
plt.tight_layout();

## 10. Полоса с дифференциальным ростом

In [ ]:
z = np.linspace(-0.5, 0.5, 401)
growth = 1.0 + 0.28 * z
strip = reduced_strip_equilibrium(z, growth)
print("lambda0:", strip["lambda0"])
print("curvature:", strip["curvature"])
print("force residual:", strip["force_residual"])
print("moment residual:", strip["moment_residual"])
fig, axes = plt.subplots(1, 2, figsize=(8, 3.8))
axes[0].plot(growth, z)
axes[0].set_xlabel("growth stretch")
axes[0].set_ylabel("z")
axes[1].plot(strip["nominal_stress"], z)
axes[1].set_xlabel("residual nominal stress")
plt.tight_layout();

## 11. Неединственность внутреннего разложения

In [ ]:
F_measured = np.diag([1.24, 0.95, 0.90])
growth_candidates = np.linspace(0.88, 1.34, 121)
energies = [total_reference_energy(F_measured, np.diag([g, 1.0, 1.0])) for g in growth_candidates]
plt.plot(growth_candidates, energies)
plt.xlabel("candidate growth stretch")
plt.ylabel("energy compatible with the same measured F")
print("same F, energy range:", min(energies), max(energies))

## 12. Синтетический верификационный benchmark

In [ ]:
benchmark = REPOSITORY_ROOT / "tutorials" / "07-growth-tensor-multiplicative-decomposition" / "data" / "growth_kinematics_benchmark.csv"
print(benchmark.read_text(encoding="utf-8"))

### Интерпретация
Прохождение этих проверок подтверждает внутреннюю согласованность учебной реализации. Оно не идентифицирует тканеспецифичный тензор роста и не валидирует биологический закон.